## To do:
- Check for NA values, other mistakes
- Clean up bill info
    - Isolate bill number?
    - identify if title has the word "Amendment"
- Connect to outside info

## Issues:
- I am able to add new columns to the dataframes, for instance to note whether a legislator voted after the vote was called, but I can't populate that column while iterating over the dataset like I'd like to. I think, in all the instances I'm trying to do this, for some reason my code is only taking the very last value it iterates over and populating the entire column, for the whole dataframe, with the same value, which is confusing/frustrating. (This can be seen most clearly with the "bill number" column I tried to create, which currently says every bill is H. 3903, which is simply not true.)

- Similarly, I want to be able to change values within already-existing columns, like when trying to remove the asterisk from late-voting legislators' names, but this isn't working either.

- I am also receiving error messages when using regexes that say things like "SyntaxWarning: invalid escape sequence '\d' --even though \d should be a perfectly reasonable thing to use within a regex, or so I thought. (I was also previously getting warnings like this over my use of * as a character per se but I seem to have solved that one.)

- My loops are also very slow and I'm sure it's just that I wrote them in a very inefficient way, but it was what made sense to me/all I could think to do when trying to write them; if you have other suggestions please do feel free to give them!!

In [7]:
import pandas as pd
import re

In [2]:
pwd

'/Users/avivafoster/ds701/ds-act-on-mass/fa25-team-a/eda/house_pdf_parsing'

In [139]:
filepath_votes = "/Users/avivafoster/ds701/ds-act-on-mass/fa25-team-a/data/parsedcsv/hjrollcallvotes_raw.csv"
df_votes = pd.read_csv(filepath_votes)

filepath_bills = "/Users/avivafoster/ds701/ds-act-on-mass/fa25-team-a/data/parsedcsv/hjbills_raw.csv"
df_bills = pd.read_csv(filepath_bills)

filepath_legs = "/Users/avivafoster/ds701/ds-act-on-mass/fa25-team-a/data/parsedcsv/MAlegislators_GC182-194.csv"
df_legislators = pd.read_csv(filepath_legs)


In [ ]:
#Check for null values in votes
for name in df_votes:
    if pd.isna(name)==True:
        print("found one")

#no current missing values, but there are empty strings! Need to convert

In [140]:
def replaceEmpty(input_str):
    if input_str == "":
        return None
    else:
        return input_str

for col in df_votes.columns:
    df_votes[col] = df_votes[col].apply(lambda x: replaceEmpty(x))


In [97]:
df_bills.replace('', None, inplace=True)

In [147]:
df_votes.dropna()
df_bills.dropna()


,bill_id,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
0,0,S. 1188 land taking,07/24/08,02:14,449,153,0,5.0
1,1,H. 4784 On Amendment,07/24/08,04:15,450,85,69,4.0
2,2,H. 4784 On Amendment,07/24/08,04:56,451,52,101,5.0
3,3,H. 4784 On Amendment,07/24/08,05:34,452,22,132,4.0
4,4,H. 4864 On Enactment,07/24/08,05:42,453,154,0,4.0
...,...,...,...,...,...,...,...,...
1918,2077,H. 3899 On Enactment Land Taking,09/30/03,02:14,416,147,0,13.0
1919,2078,QUORUM,09/30/03,02:55,417,150,0,10.0
1920,2079,S. 187 On Engrossment,09/30/03,03:10,418,146,4,10.0
1921,2080,H. 3846 On Enactment Land Taking,09/30/03,03:15,419,147,0,13.0


In [12]:
df_votes.sort_values(by = ["voter_name"]).dropna()

,bill_id,voter_name,vote_value
52580,353,*Aguiar,Y
177816,1194,*Alicea,Y
159256,1069,*Alicea,Y
169496,1138,*Alicea,Y
34504,240,*Alicea,Y
...,...,...,...
259647,1754,deMacedo,N
88127,592,deMacedo,Y
193255,1303,deMacedo,N
56615,380,deMacedo,Y


In [ ]:
#Takes a very long time to run (~over a minute) but does eventually execute... 
# though it doesn't accomplish what I wanted it to do, which is to change the values within the table
for name in df_votes["voter_name"]:
    print(name)
    asterisk_found = re.findall("[*]", str(name))
    if asterisk_found:
        new_name = re.split("[*]", str(name))[1]
        df_votes.loc[name, "voter_name"] = new_name
        df_votes["after_vote"] = 1
    else:
        df_votes["after_vote"] = 0
    

df_votes.head(20)

Webster
Fallon
Miceli
Straus
Rogers
Falzone
Moran
Sullivan
Harkins
Fennell
Murphy, C.A.
Swan
*Rushing
Fernandes
Murphy, J.
Timilty
Petrolati
Finegold
Murphy, K.J.
Tobin
Aguiar
Flanagan
Nangle
Toomey
Alicea
Flynn
Natale
Torrisi
Allen
*Forry
Naughton
Turkington
Atkins
Fox
Nyman
Turner
Atsalis
Fresolo
O'Day
Vallee
Ayers
Galvin
O'Flaherty
Verga
*Balser
Garballey
Patrick
Wagner
Basile
Garry
Peake
Wallace
Binienda
Gobi
Pedone
Walrath
Bosley
Golden
Peisch
Walsh, M.J.
Bradley
Grant
Pignatelli
Walsh, S.
Brownsberger
Greene
Provost
Walz
Cabral
Guyer
Puppolo
Welch
Callahan
Haddad
Quinn
Wolf
Calter
Hall
Reinstein
Jones
Campbell
Honan
Rice
Rogeness
Canavan
Hynes
Richardson
Peterson
Canessa
Kafka
Rodrigues
Lepper
Casey
Kane
Rush
Barrows
Clark
Kaufman
Sanchez
deMacedo
Coakley-Rivera
Keenan
Sandlin
Evangelidis
Conroy
Kennedy
Sannicandro
Frost
Costello
Khan
Scaccia
Gifford
Creedon
Kocot
Scibak
Hargraves
Curran
Koczera
Sciortino
Hill
D'Amico
Koutoujian
Smith
Humason
DeLeo
Kujawski
Smizik
Loscocco
Dempse

,bill_id,voter_name,vote_value,after_vote
0,0.0,Webster,Y,0
1,0.0,Fallon,Y,0
2,0.0,Miceli,Y,0
3,0.0,Straus,Y,0
4,0.0,Rogers,Y,0
5,0.0,Falzone,X,0
6,0.0,Moran,Y,0
7,0.0,Sullivan,Y,0
8,0.0,Harkins,Y,0
9,0.0,Fennell,Y,0


In [ ]:
#Not quite doing what I want, either... new columns are being created 
# but they're being populated with only the last value, but in every row... 
# I can't figure out how to fix it though I may just not know what to look/search for
for bill in df_bills["bill_name"]:
    bill_num = re.findall("H.*\d+\w|S.*\d+\w", str(bill))
    if bill_num:
        print(bill_num[0])
        df_bills["bill_number"] = bill_num[0]
    else:
        df_bills["bill_number"] = None
    
    amendment = re.findall("amendment", str(bill), re.IGNORECASE)
    print(amendment)
    if amendment:
        df_bills["amendment"] = 1
    else:
        df_bills["amendment"] = 0

df_bills

S. 1188
[]
H. 4784
['Amendment']
H. 4784
['Amendment']
H. 4784
['Amendment']
H. 4864
[]
S 2689
[]
[]
[]
House Rule 1A
[]
H. 4993
['Amendment']
H. 4993
[]
H 5013
[]
[]
H. 4829
[]
[]
H. 4829
[]
H. 4846
[]
[]
H. 4488
[]
H. 975
[]
S. 536
[]
H. 4740
[]
S2674
[]
S. 2468
[]
S. 2768
[]
H. 4423
['Amendment']
[]
H. 4709
[]
H. 4700
['Amendment']
H. 4700  Amendment #615
['Amendment']
H. 4700  Amendment #828
['Amendment']
H. 4700  Amendment #844
['Amendment']
H. 4700  Amendment #907
['Amendment']
H. 4700
['Amendment']
H. 4700  On Amendment     #1118
['Amendment']
H. 4700  On Amendment     #1119
['Amendment']
H. 4700
[]
[]
H. 4700
[]
H. 4700
[]
H. 4700
[]
[]
H. 4700
[]
[]
H. 4700
[]
[]
[]
H. 4700
[]
[]
H. 4700
[]
[]
H. 4700  Amendment #830
['Amendment', 'amendment']
H. 4700
[]
[]
H. 4700  Amendment #1391
['Amendment']
H. 4700
['amendment']
[]
H. 4700
['amendment']
H. 4700
[]
[]
H. 4700  AMENDMENT 472
['AMENDMENT', 'Amendment']
[]
HOUSE  rULE 1A
[]
H. 4700
[]
H. 4700
[]
[]
H. 4700
[]
H. 4700
['Amendm

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
/var/folders/41/ltdv7z254fj6rtvt1cccygrw0000gn/T/ipykernel_31872/3299597683.py:2: SyntaxWarning: invalid escape sequence '\d'
  bill_num = re.findall("H.*\d+\w|S.*\d+\w", str(bill))


,bill_id,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count,bill_number,amendment
0,0,S. 1188 land taking,07/24/08,02:14,449,153,0,5.0,H. 3903,0
1,1,H. 4784 On Amendment,07/24/08,04:15,450,85,69,4.0,H. 3903,0
2,2,H. 4784 On Amendment,07/24/08,04:56,451,52,101,5.0,H. 3903,0
3,3,H. 4784 On Amendment,07/24/08,05:34,452,22,132,4.0,H. 3903,0
4,4,H. 4864 On Enactment,07/24/08,05:42,453,154,0,4.0,H. 3903,0
...,...,...,...,...,...,...,...,...,...,...
1918,2077,H. 3899 On Enactment Land Taking,09/30/03,02:14,416,147,0,13.0,H. 3903,0
1919,2078,QUORUM,09/30/03,02:55,417,150,0,10.0,H. 3903,0
1920,2079,S. 187 On Engrossment,09/30/03,03:10,418,146,4,10.0,H. 3903,0
1921,2080,H. 3846 On Enactment Land Taking,09/30/03,03:15,419,147,0,13.0,H. 3903,0
